# 小說三聲線有聲書｜免費 Colab 版

使用你有權轉製的小說檔案，產生 **旁白（女聲）／宋辭（男聲）／許明珠（女聲）** MP3。

**步驟：** 依序執行 1、2，下載角色校對 CSV，修改不確定的對話角色，再執行 3。首次執行需網路，Edge TTS 是線上服務，可能被限流，不保證永遠免費；不是原作者的聲音，也不保證有戲劇化情緒效果。

**隱私：** 不要將未授權公開的小說文本提交到 GitHub；只在 Colab 暫時上傳。掃描 PDF 可能無法擷取，角色辨識須人工校對。

In [ ]:
# @title 1. 安裝依賴（按 ▶）
!pip -q install edge-tts ebooklib beautifulsoup4 python-docx pypdf pydub pandas
!apt-get -qq install -y ffmpeg > /dev/null


In [ ]:
# @title 2. 上傳小說並製作角色校對表（按 ▶）
import os, re, io, pathlib, zipfile, asyncio, hashlib, html
import pandas as pd
from google.colab import files
from bs4 import BeautifulSoup
from ebooklib import epub, ITEM_DOCUMENT
from pypdf import PdfReader
from docx import Document
from pydub import AudioSegment
from IPython.display import display
WORK='/content/novel_audio'
os.makedirs(WORK, exist_ok=True)
uploaded=files.upload()
assert len(uploaded)==1, "請一次上傳一本小說"
filename, raw=next(iter(uploaded.items()))
ext=pathlib.Path(filename).suffix.lower()
source=os.path.join(WORK, filename)
with open(source,'wb') as f: f.write(raw)
if ext=='.txt':
    try: text=raw.decode('utf-8-sig')
    except UnicodeDecodeError: text=raw.decode('big5', errors='replace')
elif ext=='.epub':
    book=epub.read_epub(source)
    text='\n'.join(BeautifulSoup(x.get_content(),'html.parser').get_text('\n') for x in book.get_items() if x.get_type()==ITEM_DOCUMENT)
elif ext=='.pdf':
    reader=PdfReader(io.BytesIO(raw))
    text='\n'.join((p.extract_text() or '') for p in reader.pages)
elif ext=='.docx':
    text='\n'.join(p.text for p in Document(io.BytesIO(raw)).paragraphs)
else:
    raise ValueError('僅支援 TXT、EPUB、PDF、DOCX')
text=re.sub(r'\r\n?', '\n', text).strip()
assert len(text)>30, "檔案沒有足夠的可擷取文字，掃描型 PDF 不適用"
# @markdown 從第幾章開始？（若原文沒有可辨識章名，使用全文）
start_chapter=151 # @param {type:"integer"}
# @markdown 最多處理多少個章節？（先使用 1 章，確認品質後可增加）
chapter_count=1 # @param {type:"integer"}
pattern=re.compile(r'(?m)^\s*(?:第\s*(\d{1,5})\s*章|Chapter\s+(\d{1,5})\b)[^\n]*$',re.I)
matches=list(pattern.finditer(text))
selected=text
if matches:
    starts=[(int(m.group(1) or m.group(2)),m.start()) for m in matches]
    idx=next((i for i,(num,_) in enumerate(starts) if num==start_chapter),None)
    if idx is None: print('找不到指定章節標題，改用全文；請先檢查來源。')
    else:
        a=starts[idx][1]; b=starts[min(idx+chapter_count,len(starts)-1)][1] if idx+chapter_count<len(starts) else len(text)
        selected=text[a:b]
else:
    print('未辨識到章節標題；將使用全文。')
# 以段落和引號切分。對話歸屬只能做初判，需手動校對 speaker 欄。
paragraphs=[p.strip() for p in re.split(r'\n+',selected) if p.strip()]
parts=[]
for p in paragraphs:
    chunks=re.split(r'([「“][^」”]{1,200}[」”])',p)
    for chunk in chunks:
        if not chunk.strip(): continue
        quoted=bool(re.fullmatch(r'[「“].*[」”]',chunk,re.S))
        # 只有明確緊接姓名與說話動詞時才自動歸屬，否則標為未確認
        context=parts[-1]['text'][-45:] if parts else ''
        name=None
        if quoted and re.search(r'宋辭.{0,8}(?:說|問|道|笑道|開口)',context): name='宋辭'
        elif quoted and re.search(r'許明珠.{0,8}(?:說|問|道|笑道|開口)',context): name='許明珠'
        parts.append(dict(speaker=name if name else ('待確認對話' if quoted else '旁白'),text=chunk))
# 避免 TTS 單段過長；保留文本順序
rows=[]
for item in parts:
    s=item['text']
    for i in range(0,len(s),220):
        rows.append(dict(speaker=item['speaker'],text=s[i:i+220]))
df=pd.DataFrame(rows)
csv=os.path.join(WORK,'角色校對表.csv')
df.to_csv(csv,index=False,encoding='utf-8-sig')
print('分段數：',len(df),'；需要確認的對話：',sum(df.speaker=='待確認對話'))
display(df.head(30))
files.download(csv)
print('請在 CSV 將待確認對話改成 宋辭／許明珠／旁白／其他男聲／其他女聲，再執行下一格。')


In [ ]:
# @title 3. 上傳校對後 CSV，免費產生 MP3（按 ▶）
import edge_tts, time
from google.colab import files
from pydub import AudioSegment
# @markdown 允許對話尚未確認時暫時由旁白朗讀？
allow_unconfirmed=False # @param {type:"boolean"}
# @markdown 語速調整（0 為一般速度）
rate_percent=-5 # @param {type:"integer"}
VOICE={'旁白':'zh-TW-HsiaoChenNeural','宋辭':'zh-TW-YunJheNeural','許明珠':'zh-TW-HsiaoYuNeural',
       '其他男聲':'zh-TW-YunJheNeural','其他女聲':'zh-TW-HsiaoChenNeural'}
csv_upload=files.upload()
assert len(csv_upload)==1, '請上傳剛剛校對好的 CSV'
data=next(iter(csv_upload.values()))
df=pd.read_csv(io.BytesIO(data),encoding='utf-8-sig').fillna('')
assert {'speaker','text'}.issubset(df.columns)
unknown=set(df.speaker)-set(VOICE)
if unknown and not (allow_unconfirmed and unknown=={'待確認對話'}):
    raise ValueError('未完成角色校對：'+str(unknown))
df.speaker=df.speaker.replace({'待確認對話':'旁白'})
output_dir=os.path.join(WORK,'segments');os.makedirs(output_dir,exist_ok=True)
async def synthesize(i, speaker, text):
    if not str(text).strip():return None
    key=hashlib.sha256((speaker+text+str(rate_percent)).encode()).hexdigest()[:14]
    path=os.path.join(output_dir,f'{i:05d}_{key}.mp3')
    if not os.path.exists(path):
        for attempt in range(3):
            try:
                tts=edge_tts.Communicate(text, VOICE[speaker], rate=f'{rate_percent:+d}%')
                await tts.save(path)
                break
            except Exception:
                if attempt==2: raise
                await asyncio.sleep(2*(attempt+1))
    return path
async def run():
    paths=[]
    for i,row in df.iterrows():
        path=await synthesize(i,row.speaker,str(row.text))
        if path: paths.append(path)
        if (i+1)%20==0:print('已生成',i+1,'/',len(df))
    return paths
paths=await run()
audio=AudioSegment.empty()
for p in paths:audio+=AudioSegment.from_mp3(p)+AudioSegment.silent(duration=220)
out=os.path.join(WORK,f'第{start_chapter}章起_{chapter_count}章_三聲線.mp3')
audio.export(out,format='mp3',bitrate='128k')
print('完成：',out,'；長度（分鐘）：',round(len(audio)/60000,1))
files.download(out)


## 進階版

需要更高品質本地模型時，參考 [ebook2audiobook 官方專案](https://github.com/DrewThomasson/ebook2audiobook)。這份筆記本屬於自訂 Edge TTS 三聲線工作流，不宣稱為 ebook2audiobook 原生功能。